In [1]:
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import pandas as pd
import seaborn as sns

In [3]:
df = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

In [4]:
from sklearn.model_selection import train_test_split
X = df.iloc[:,:-1]
Y = df.iloc[:,-1]
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.1, random_state = 42)

In [5]:
def scale_with_names(XTrain, XTest, scaler = None):
    if scaler is None:
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
    arr1 = scaler.fit_transform(XTrain)
    XTrain = pd.DataFrame(arr1, columns = XTrain.columns, index = XTrain.index)
    arr2 = scaler.transform(XTest)
    XTest = pd.DataFrame(arr2, columns = XTest.columns, index = XTest.index)
    return XTrain, XTest



In [6]:
# Now we can just perform Min Max Scaling on the TrackDuration Column

df_new = X_train[['TrackDurationMs']]
df_new_test = df_test[['TrackDurationMs']]

df_new, df_new_test = scale_with_names(df_new, df_new_test)

X_train['TrackDurationMs'] = df_new['TrackDurationMs']
df_test['TrackDurationMs'] = df_new_test['TrackDurationMs']





In [7]:
# Now we need to decide upon the Model that we are supposed to use
"""
We can go for many Regression Models, Lasso, Ridge and even Linear.
We will go through them all evaluating the model on the basis of the Loss function defined in the contest
and evaluate the model finally using RandomForestRegressor as well
"""

# Let's first use Ridge Regression

from sklearn.metrics import mean_squared_error

from sklearn.metrics import mean_absolute_percentage_error


from sklearn.linear_model import Ridge
model = Ridge()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred)

print("RMSE : ", rmse)

mape = mean_absolute_percentage_error(y_test, y_pred)

print("MAPE: ", mape)





RMSE :  76498171167.671
MAPE:  2387.830795404278


In [8]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import BaggingRegressor

base_tree = DecisionTreeRegressor(random_state = 42)
bagging = BaggingRegressor(estimator = base_tree,
                           random_state = 42,
                           n_jobs = -1
)





In [9]:
param_grids ={
    'n_estimators': [50,20,5],
    'max_samples': [0.5,0.8,1.0],
    'estimator__max_depth':[None,],
    'estimator__min_samples_split': [2]
}


In [10]:
grid_search = GridSearchCV(
    estimator = bagging,
    param_grid = param_grids,
    scoring = "neg_mean_squared_error",
    cv = 5,
    n_jobs = -1
)

In [11]:
 #grid_search.fit(X_train, y_train)

In [12]:
# y_pred = grid_search.predict(df_test)

In [13]:
'''submission = pd.read_csv("sample_submission.csv")
submission["BeatsPerMinute"] = y_pred
'''

'submission = pd.read_csv("sample_submission.csv")\nsubmission["BeatsPerMinute"] = y_pred\n'

In [14]:
# submission.to_csv("DT_Regressor.csv", index = False)

In [15]:
# Now we need to apply XG Boost here

In [16]:
from sklearn.ensemble import GradientBoostingRegressor

boostreg = GradientBoostingRegressor(loss = 'squared_error', learning_rate = 0.001, n_estimators = 1000, subsample = 0.9, min_samples_leaf = 15, verbose = 1)

In [17]:
'''X_fit = boostreg.fit(X_train, y_train)


from sklearn.metrics import mean_squared_error
y_pred = X_fit.predict(X_test)

rmse = mean_squared_error(y_test, y_pred)

print("RMSE : ", rmse)
'''

'X_fit = boostreg.fit(X_train, y_train)\n\n\nfrom sklearn.metrics import mean_squared_error\ny_pred = X_fit.predict(X_test)\n\nrmse = mean_squared_error(y_test, y_pred)\n\nprint("RMSE : ", rmse)\n'

In [18]:
 # y_pred = X_fit.predict(df_test)

In [19]:
'''submission = pd.read_csv("sample_submission.csv")
submission["BeatsPerMinute"] = y_pred
'''

'submission = pd.read_csv("sample_submission.csv")\nsubmission["BeatsPerMinute"] = y_pred\n'

In [20]:
 #submission.to_csv("Boosting_Regressor3.csv", index = False)

In [21]:
'''from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.linear_model import LinearRegression

model  = Ridge()
model2 = Lasso()
model3 = RandomForestRegressor()
model4 =
 BaggingRegressor(estimator = None, n_estimators = 100, max_samples = 0.5)
model5 = LinearRegression()

model5.fit(X_train, y_train)
y_pred = model5.predict(df_test)
'''


'from sklearn.linear_model import Ridge\nfrom sklearn.linear_model import Lasso\nfrom sklearn.ensemble import RandomForestRegressor\nfrom sklearn.ensemble import BaggingRegressor\nfrom sklearn.linear_model import LinearRegression\n\nmodel  = Ridge()\nmodel2 = Lasso()\nmodel3 = RandomForestRegressor()\nmodel4 =\n BaggingRegressor(estimator = None, n_estimators = 100, max_samples = 0.5)\nmodel5 = LinearRegression()\n\nmodel5.fit(X_train, y_train)\ny_pred = model5.predict(df_test)\n'

In [22]:
import xgboost as xgb


In [23]:
xgb_model = xgb.XGBRegressor(n_estimators = 1000, max_depth = 6, grow_policy = 'depthwise', learning_rate = 0.001, verbosity = 1, objective = 'reg:squarederror', booster = 'gbtree', tree_method = 'gpu_hist', n_jobs = -1, subsample = 0.9, random_state = 42)


xgb_model.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [13:46:24] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)


XGBRegressor(base_score=None, booster='gbtree', callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy='depthwise',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.001, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [24]:
y_pred = xgb_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

mse

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [13:46:28] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:729: UserWarning: [13:46:28] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


699.115063311915

In [25]:
y_pred = xgb_model.predict(df_test)



In [26]:
submission = pd.read_csv("sample_submission.csv")
submission["BeatsPerMinute"] = y_pred

In [27]:
submission.to_csv("XGBoosting_Regressor2.csv", index = False)

In [28]:
xgb_model

XGBRegressor(base_score=None, booster='gbtree', callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy='depthwise',
             importance_type=None, interaction_constraints=None,
             learning_rate=0.001, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)